# Предсказание стоимости жилья

В проекте вам нужно обучить модель линейной регрессии на данных о жилье в Калифорнии в 1990 году. На основе данных нужно предсказать медианную стоимость дома в жилом массиве. Обучите модель и сделайте предсказания на тестовой выборке. Для оценки качества модели используйте метрики RMSE, MAE и R2.

## Импорты

In [1]:
import pyspark
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import *
from pyspark.sql.types import NumericType

pyspark_version = pyspark.__version__
print("pyspark version:", pyspark_version)

if int(pyspark_version[:1]) == 3:
    from pyspark.ml.feature import OneHotEncoder
elif int(pyspark_version[:1]) == 2:
    from pyspark.ml.feature import OneHotEncodeEstimator

RANDOM_SEED = 1


pyspark version: 3.0.2


## Функции

In [2]:
import os

HOST = "https://code.s3.yandex.net"
HTTP_PREFIX = "http"


# load csv
def load_csv(spark: SparkSession, dataset_path: str, **kwargs):
    # check server request --> relative path --> absolute path --> yandex server request
    path = (
        dataset_path
        if dataset_path.startswith(HTTP_PREFIX)
        else "." + dataset_path if os.path.exists("." + dataset_path)
        else dataset_path if os.path.exists(dataset_path)
        else HOST + dataset_path
    )
    print("Dataset path:", path)
    try:
        return spark.read.option('header', 'true').csv(path=path, **kwargs)
    except Exception as ex:
        print("Could not load csv. Exception:", str(ex))

## Подготовка данных

In [3]:
spark = SparkSession.builder \
    .master("local") \
    .appName("California Housing") \
    .getOrCreate()

In [4]:
df_housing = load_csv(spark, dataset_path="/datasets/housing.csv", inferSchema=True)
df_housing.show(10)

Dataset path: /datasets/housing.csv


+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|
|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|
|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|
|  -122.25|   37.85|              

In [5]:
df_housing.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)
 |-- ocean_proximity: string (nullable = true)



In [6]:
df_housing.summary().show()

+-------+-------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+---------------+
|summary|          longitude|         latitude|housing_median_age|       total_rooms|    total_bedrooms|        population|       households|     median_income|median_house_value|ocean_proximity|
+-------+-------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+---------------+
|  count|              20640|            20640|             20640|             20640|             20433|             20640|            20640|             20640|             20640|          20640|
|   mean|-119.56970445736148| 35.6318614341087|28.639486434108527|2635.7630813953488| 537.8705525375618|1425.4767441860465|499.5396802325581|3.8706710029070246|206855.81690891474|           null|
| stddev|  2.0035317

Большое значение stddev для целевого признака говорит о широком разбросе данных вокруг среднего значения.

Присутствуют пропуски значений в колонке `total_bedrooms`. Заполним их медианным значением.

In [7]:
# Расчет медианного значения для колонки "total_bedrooms":
median_total_bedrooms = df_housing.approxQuantile("total_bedrooms", [0.5], 0)[0]

# Замещение пропусков медианой
df_housing_filled = df_housing.fillna(median_total_bedrooms, subset=["total_bedrooms"])

# Проверяем результат
df_housing_filled.summary().show()

+-------+-------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+---------------+
|summary|          longitude|         latitude|housing_median_age|       total_rooms|    total_bedrooms|        population|       households|     median_income|median_house_value|ocean_proximity|
+-------+-------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+---------------+
|  count|              20640|            20640|             20640|             20640|             20640|             20640|            20640|             20640|             20640|          20640|
|   mean|-119.56970445736148| 35.6318614341087|28.639486434108527|2635.7630813953488| 536.8388565891473|1425.4767441860465|499.5396802325581|3.8706710029070246|206855.81690891474|           null|
| stddev|  2.0035317

Выделим числовые колонки в датасете. Это понадобится на этапе обучения и сравнения метрик моделей. Исключим из списка целевой признак `median_house_value`.

In [8]:
# Выделяем числовые колонки
num_columns = [f.name for f in df_housing.schema.fields
               if isinstance(f.dataType, NumericType) and not f.name == "median_house_value"]
num_columns

['longitude',
 'latitude',
 'housing_median_age',
 'total_rooms',
 'total_bedrooms',
 'population',
 'households',
 'median_income']

Преобразуем колонку `ocean_proximity` с категориальными значениями техникой One hot encoding

In [9]:
# индексирование категориальных признаков
indexer = StringIndexer(inputCol="ocean_proximity", outputCol="ocean_proximity_index")
model = indexer.fit(df_housing_filled)
df_housing_filled_indexed = model.transform(df_housing_filled)

# One-Hot Encoding
encoder = OneHotEncoder(inputCols=['ocean_proximity_index'], outputCols=['ocean_proximity_vector'])
df_housing_filled_encoded = encoder.fit(df_housing_filled_indexed).transform(df_housing_filled_indexed)

df_housing_filled_encoded.show(10)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------------+----------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|ocean_proximity_index|ocean_proximity_vector|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------------+----------------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|                  3.0|         (4,[3],[1.0])|
|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|                  3.0|         (4,[3],[1.0])|
|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177

Выделим числовые колонки в датасете после преобразования категориального признака. Это понадобится на этапе обучения и сравнения метрик моделей. Исключим из списка целевой признак `median_house_value`.

In [10]:
num_columns_encoded = [f.name for f in df_housing_filled_encoded.schema.fields
                       if not isinstance(f.dataType, StringType) and not f.name == "median_house_value"]
num_columns_encoded

['longitude',
 'latitude',
 'housing_median_age',
 'total_rooms',
 'total_bedrooms',
 'population',
 'households',
 'median_income',
 'ocean_proximity_index',
 'ocean_proximity_vector']

Данные подготовлены. Перейдем к обучению моделей линейной регрессии.

## Обучение моделей

Построим две модели линейной регрессии на разных наборах данных:
- используя все данные из файла
- используя только числовые переменные, исключив категориальные

Для построения моделей будем использовать оценщик LinearRegression из библиотеки MLlib.

Создадим функцию для обучения модели линейной регресии и вывода метрик оценки качества полученной модели.

In [11]:
def get_model_with_metrics(df: DataFrame, features: list, target: str):
    """
    Обучает модель линейной регрессии и выводит метрики оценки качества полученной модели (RMSE, MAE, R2)

    :param df: датасет для обучения
    :param features: список колонок с признаками для обучения
    :param target: целевой признак
    :return: модель линейной регрессии
    """
    # Объединяем признаки в единый вектор признаков
    assembler = VectorAssembler(
        inputCols=features,
        outputCol="features"
    )
    assembled_data = assembler.transform(df)

    # Определяем и настраиваем стандартный скейлер
    scaler = StandardScaler(
        inputCol="features",
        outputCol="scaled_features",
        withStd=True,
        withMean=True
    )

    # Выполним fit и transform над данными
    scaled_data = scaler.fit(assembled_data).transform(assembled_data)

    # Настройка и создание модели линейной регрессии
    lr = LinearRegression(
        featuresCol="scaled_features",
        labelCol=target,
        regParam=0.1
    )

    # Обучаем модель на тренировочных данных
    model = lr.fit(scaled_data)

    # Делаем предсказания
    predictions = model.transform(scaled_data)

    # Оценим качество модели
    evaluator_rmse = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="rmse")
    rmse = evaluator_rmse.evaluate(predictions)

    evaluator_mae = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="mae")
    mae = evaluator_mae.evaluate(predictions)

    evaluator_r2 = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="r2")
    r2 = evaluator_r2.evaluate(predictions)

    # Выведем полученные метрики
    print("Root Mean Squared Error (RMSE):", rmse)
    print("Mean Absolute Error (MAE):", mae)
    print("Coefficient of Determination (R2):", r2)

    return model

Построим наши модели и оценим их качество с помощью определенной выше функции. В функцию передадим физическую копию датасетов, полученную с использованием repartition().

In [12]:
get_model_with_metrics(df_housing_filled.repartition(10), num_columns, "median_house_value")

25/10/29 21:02:49 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
25/10/29 21:02:49 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS
25/10/29 21:02:50 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeSystemLAPACK
25/10/29 21:02:50 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeRefLAPACK


Root Mean Squared Error (RMSE): 69658.19035715944
Mean Absolute Error (MAE): 50922.85399198825
Coefficient of Determination (R2): 0.6355929262652602


LinearRegressionModel: uid=LinearRegression_f83b2110193f, numFeatures=8

In [13]:
get_model_with_metrics(df_housing_filled_encoded.repartition(10), num_columns_encoded, "median_house_value")

Root Mean Squared Error (RMSE): 68709.32559470677
Mean Absolute Error (MAE): 49828.74105086885
Coefficient of Determination (R2): 0.645453016428341


LinearRegressionModel: uid=LinearRegression_b0d4b51c3559, numFeatures=13

In [14]:
spark.stop()

## Анализ результатов

Показатели полученных моделей для рзаного количества признаков не сильно отличаются.

Стандартное отклонение для колонки `median_house_value` составляет примерно 115396, что говорит о широком разбросе данных вокруг среднего значения.

Показатели MAE для моделей не сильно отличаются друг от друга. Модель в среднем отклоняется от реальной цены примерно на 50000, что выглядит довольно существенным, т.к. составляет 10% от максимальной стоимости (500001) и почти 25% от значения средней стоимости (206856). А если говорить о медианном значении цены (179700), то средняя ошибка будет больше 25%.

Показатель R2 для обеих моделей тоже не сильно отличается и составляет примерно 0.64, что говорит о посредственном качестве моделей. Они лишь частично объясняют вариацию цены.